## 🎯 Learning Objectives
* Understand the critical role of human feedback in developing robust, production-grade AI agents.
* Identify different mechanisms for integrating human feedback into CrewAI workflows.
* Implement a simulated human-in-the-loop feedback mechanism within a CrewAI application.
* Analyze the trade-offs and benefits of incorporating human feedback for continuous agent improvement.
* Explore how collected human feedback can inform prompt engineering, tool refinement, and agent behavior adaptation.


## Training Your Crew with Human Feedback: The Human-in-the-Loop Advantage

In the rapidly evolving landscape of AI, building agents that perform reliably and align perfectly with complex business objectives remains a significant challenge. While large language models (LLMs) are incredibly powerful, they often struggle with nuance, edge cases, and the dynamic nature of real-world tasks. This is where **human feedback** becomes indispensable, transforming a capable AI agent into a truly intelligent and trustworthy assistant.

Imagine a junior analyst joining your team. Initially, they might produce reports that are technically correct but lack the specific tone, depth, or focus your organization requires. As a senior manager, you provide feedback: "This section needs more detail on market trends," or "The conclusion should be more actionable." Over time, the junior analyst learns from these corrections, refining their approach and eventually producing work that consistently meets your standards. 

AI agents, particularly those orchestrated by frameworks like CrewAI, benefit from a similar learning process. **Human-in-the-Loop (HITL)** systems allow human experts to review, correct, and guide agent behavior, ensuring that the AI's outputs are not just plausible, but truly valuable and aligned with business goals. This is crucial for production environments where errors can have significant financial or reputational consequences.

### Why Human Feedback is Critical for Production Agents:

1.  **Alignment with Business Objectives:** LLMs are trained on vast datasets, but they don't inherently understand your company's specific strategies, values, or compliance requirements. Human feedback bridges this gap.
2.  **Handling Edge Cases:** Real-world scenarios are messy. Agents might excel at common tasks but falter on unusual inputs. Human review helps identify and correct these failures, making the agent more robust.
3.  **Improving Reliability and Trust:** Consistent, high-quality outputs build trust. When agents make mistakes, human correction prevents recurrence, leading to more reliable automation.
4.  **Adapting to Change:** Business environments, regulations, and market conditions evolve. Human feedback allows agents to adapt quickly without requiring full model retraining.
5.  **Ethical and Safety Guardrails:** Humans can identify and correct biased, harmful, or inappropriate agent behaviors, ensuring responsible AI deployment.

### How Human Feedback Integrates with CrewAI:

In a CrewAI context, integrating human feedback typically involves:

*   **Dedicated Human Reviewer Agent:** You can design an agent whose primary role is to present outputs to a human, collect feedback, and then relay that feedback back into the crew's workflow.
*   **Feedback Collection Mechanisms:** This could range from simple approval/rejection buttons in a UI, detailed text corrections, preference comparisons (e.g., "Which of these two outputs is better?"), or even demonstrations of correct behavior.
*   **Feedback Utilization:** The collected feedback can be used in several ways:
    *   **Prompt Engineering:** Directly modify or refine the prompts given to agents for future tasks.
    *   **Tool Selection/Usage Refinement:** Guide agents on when and how to use specific tools more effectively.
    *   **Agent Behavior Adjustment:** Update an agent's internal `backstory` or `goal` to reflect new learnings.
    *   **Data for Fine-tuning (Offline):** For more advanced scenarios, structured human feedback can be collected as a dataset to fine-tune smaller, specialized LLMs or adapt larger models using techniques like Reinforcement Learning from Human Feedback (RLHF) or Direct Preference Optimization (DPO). While full RLHF/DPO is complex, the *collection* of high-quality human feedback is the first step.

In the following example, we'll simulate a simplified human feedback loop within a CrewAI setup. An "Analyst" agent will generate a summary, and a "Human Reviewer" agent (simulated by a custom tool) will provide feedback, which can then conceptually be used to improve the Analyst's future performance.


In [ ]:
import os
from crewai import Agent, Task, Crew, Process
from crewai_tools import Tool
from typing import List, Dict

# --- 2026 Ready: Mock LLM for demonstration without API keys ---
# In a real 2026 production scenario, you'd use a robust LLM provider
# like OpenAI's GPT-4o, Google's Gemini 1.5 Pro, Anthropic's Claude 3.5 Sonnet,
# or a self-hosted model like Llama 4 or Mistral Large via Ollama/vLLM.
# For this example, we'll simulate LLM responses.

class MockLLM:
    def __init__(self, responses: Dict[str, str]):
        self.responses = responses
        self.call_count = 0

    def invoke(self, prompt: str) -> str:
        self.call_count += 1
        print(f"\n--- MockLLM Call {self.call_count} ---")
        print(f"Prompt: {prompt[:200]}...") # Print first 200 chars of prompt
        
        # Simple keyword-based response simulation
        if "summarize" in prompt.lower():
            return self.responses.get("summary", "This is a mock summary of the provided text.")
        elif "review" in prompt.lower() and "summary" in prompt.lower():
            return self.responses.get("review", "The summary is good, but could be more concise and highlight key financial implications.")
        elif "refine" in prompt.lower() and "summary" in prompt.lower():
            return self.responses.get("refined_summary", "Refined summary: Key financial implications are now clearly highlighted, making it more concise.")
        else:
            return self.responses.get("default", "Mock LLM response for: " + prompt[:50] + "...")

# Initialize mock LLM with predefined responses
mock_llm_responses = {
    "summary": "The Q1 2026 earnings report shows a 15% revenue increase driven by AI infrastructure sales, but profit margins were slightly down due to increased R&D spending. The outlook for Q2 is positive, projecting continued growth in cloud services.",
    "review": "Feedback: The summary is accurate, but it lacks specific financial figures and doesn't clearly state the impact on shareholder value. Please refine to include these details and make it more actionable for investors. Rating: 3/5.",
    "refined_summary": "Refined Q1 2026 Earnings Summary: Revenue surged by 15% to $1.2B, primarily from AI infrastructure. Despite a 2% dip in profit margins due to $200M R&D investment, EPS remained strong at $1.50. This growth trajectory, coupled with a positive Q2 outlook for cloud services, suggests strong potential for shareholder value appreciation. Rating: 5/5."
}
mock_llm = MockLLM(mock_llm_responses)

# --- Custom Tool for Human Feedback Simulation ---
# In a real scenario, this tool would interact with a UI or a human feedback platform.

def get_human_feedback(agent_output: str) -> str:
    """Simulates receiving human feedback on an agent's output."""
    print(f"\n--- Awaiting Human Feedback ---")
    print(f"Agent Output for Review:\n{agent_output}")
    
    # In a real application, this would be an API call to a human review system
    # or a prompt for user input in a CLI/GUI.
    
    # For demonstration, we'll use a predefined mock feedback.
    feedback = mock_llm_responses["review"]
    print(f"Simulated Human Feedback Received: {feedback}")
    return feedback

human_feedback_tool = Tool(
    name="Human Feedback Tool",
    func=get_human_feedback,
    description="Collects feedback from a human reviewer on an agent's output."
)

# --- Define Agents ---

# Agent 1: The Analyst - generates initial content
analyst_agent = Agent(
    role='Financial Analyst',
    goal='Summarize complex financial reports and identify key insights for investors.',
    backstory='An expert in financial markets, skilled at distilling dense information into actionable intelligence.',
    llm=mock_llm,
    verbose=True,
    allow_delegation=False
)

# Agent 2: The Human Reviewer (simulated) - provides feedback
# This agent's primary 'intelligence' comes from the human_feedback_tool
# which simulates human input.
reviewer_agent = Agent(
    role='Human Feedback Integrator',
    goal='Review agent outputs for accuracy, completeness, and alignment with business objectives, then provide constructive feedback.',
    backstory='A critical eye, ensuring all automated outputs meet the highest standards before finalization. Acts as a bridge between AI and human expertise.',
    llm=mock_llm, # Still needs an LLM for its own reasoning, but feedback comes from tool
    verbose=True,
    allow_delegation=False,
    tools=[human_feedback_tool]
)

# --- Define Tasks ---

# Task 1: Initial Summary Generation
task_summarize = Task(
    description=(
        "Analyze the provided Q1 2026 earnings report data and generate a concise summary. "
        "Focus on revenue growth, profit margins, key drivers, and future outlook. "
        "Ensure the summary is suitable for a high-level investor briefing." 
        "Data: Q1 2026 Earnings Report - Revenue: $1.2 Billion (up 15% YoY), Profit Margin: 18% (down 2% YoY), "
        "Key Drivers: AI Infrastructure Sales, Cloud Services Growth. R&D Spending: $200 Million (up 25% YoY). "
        "EPS: $1.50. Outlook: Positive for Q2, continued cloud services expansion."
    ),
    agent=analyst_agent,
    expected_output="A 3-4 sentence summary of the Q1 2026 earnings report."
)

# Task 2: Human Feedback Collection
task_get_feedback = Task(
    description=(
        "Review the initial summary generated by the Financial Analyst. "
        "Use the 'Human Feedback Tool' to simulate collecting feedback from a human expert. "
        "The feedback should assess accuracy, completeness, and investor relevance. "
        "Pass the summary to the tool and capture the feedback provided." 
        "Input Summary: {initial_summary}"
    ),
    agent=reviewer_agent,
    expected_output="A detailed string containing human feedback and a rating for the initial summary."
)

# Task 3: Refine Summary based on Feedback
task_refine_summary = Task(
    description=(
        "Based on the human feedback received, refine the initial financial summary. "
        "Address all points raised in the feedback to improve accuracy, conciseness, "
        "and investor relevance. Ensure specific financial figures and shareholder impact are included. "
        "Initial Summary: {initial_summary}\nHuman Feedback: {feedback}"
    ),
    agent=analyst_agent,
    expected_output="A significantly improved and refined 3-5 sentence financial summary addressing all feedback points."
)

# --- Create the Crew ---

# Define the crew with a sequential process
# The output of one task feeds into the next.
crew = Crew(
    agents=[analyst_agent, reviewer_agent],
    tasks=[task_summarize, task_get_feedback, task_refine_summary],
    process=Process.sequential,
    verbose=2 # Set to 1 for less verbose, 2 for full output
)

# --- Kick off the Crew's work ---
print("\n### Starting the Crew's Work with Human Feedback Loop ###")

# Execute the crew. The outputs of previous tasks are automatically passed
# as context to subsequent tasks when using sequential process.
# We explicitly pass the initial summary to the feedback task for clarity.

# First, run the initial summary task to get its output
initial_summary_result = crew.tasks[0].execute()

# Now, run the feedback task, passing the initial summary
feedback_result = crew.tasks[1].execute(context={'initial_summary': initial_summary_result})

# Finally, run the refine task, passing both the initial summary and the feedback
final_refined_summary = crew.tasks[2].execute(context={
    'initial_summary': initial_summary_result,
    'feedback': feedback_result
})

print("\n### Crew Work Completed ###")
print(f"\nInitial Summary:\n{initial_summary_result}")
print(f"\nHuman Feedback Received:\n{feedback_result}")
print(f"\nFinal Refined Summary:\n{final_refined_summary}")

# --- Conceptualizing Continuous Improvement ---
# In a real system, 'feedback_result' would be stored in a database.
# This data could then be used:
# 1. To update the 'backstory' or 'goal' of the 'analyst_agent' for future runs.
# 2. To dynamically adjust the prompts for the 'analyst_agent' based on common feedback patterns.
# 3. As training data for fine-tuning a smaller, specialized LLM for summarization.

# Example of how feedback could conceptually update an agent's prompt for future tasks:
# if "more concise" in feedback_result.lower():
#     analyst_agent.goal = 'Summarize complex financial reports concisely, highlighting key insights for investors.'
#     print("\nAnalyst Agent's goal updated based on feedback!")


### Interpreting the Code Output and Practical Implications

The code above demonstrates a simplified, conceptual human-in-the-loop feedback mechanism within a CrewAI workflow. Let's break down what's happening and its real-world implications:

1.  **Mock LLM:** We use a `MockLLM` to simulate the behavior of a real LLM. In 2026, you would replace this with actual API calls to advanced models like GPT-4o, Gemini 1.5 Pro, or a fine-tuned open-source model. The mock allows us to control responses and focus on the feedback loop logic without external dependencies.

2.  **`Human Feedback Tool`:** This is the core of our human-in-the-loop simulation. In a production environment, `get_human_feedback` would not return a hardcoded string. Instead, it would:
    *   Send the `agent_output` to a dedicated human review platform (e.g., Argilla, Label Studio, a custom internal UI).
    *   Wait for a human expert to review the output and provide structured feedback (e.g., a rating, specific corrections, or a revised version).
    *   Return that human-provided feedback to the CrewAI agent.

3.  **`Reviewer Agent`:** This agent's role is to orchestrate the feedback collection. It uses the `Human Feedback Tool` to interact with the simulated human. Its `llm` is still used for its own reasoning (e.g., deciding *when* to use the tool, or how to interpret the tool's output), but the *content* of the feedback comes from the tool.

4.  **Sequential Process:** The `Crew` is set up with `Process.sequential`. This is crucial because it ensures that the output of one task (the initial summary) becomes the input for the next task (getting human feedback), and then both become input for the final task (refining the summary).

5.  **Output Interpretation:**
    *   **Initial Summary:** You'll see the first attempt by the `Financial Analyst` agent to summarize the data.
    *   **Human Feedback Received:** This output, simulated by our `Human Feedback Tool`, represents the critical input from a human expert. Notice how it points out specific areas for improvement (e.g., "lacks specific financial figures," "doesn't clearly state the impact on shareholder value").
    *   **Final Refined Summary:** This shows the `Financial Analyst` agent incorporating the human feedback to produce a significantly improved summary, addressing the identified shortcomings. This demonstrates the *immediate* impact of feedback on agent performance.

### Performance Trade-offs and Use Cases:

**Performance Trade-offs:**

*   **Latency:** Introducing a human into the loop inevitably adds latency. Human review takes time, which might not be acceptable for real-time applications. For asynchronous tasks or those where accuracy is paramount, this trade-off is often acceptable.
*   **Cost:** Human labor is a cost. The more frequently humans are involved, the higher the operational cost. Strategies often involve sampling (reviewing a subset of outputs) or focusing human review on high-stakes tasks or new agent deployments.
*   **Scalability:** Relying too heavily on human review can create a bottleneck, limiting the scalability of your automated processes.

**Typical Use Cases for Human Feedback in Production CrewAI Systems (2026):**

*   **Compliance and Legal Review:** Agents drafting legal documents or compliance reports *must* be accurate. Human lawyers or compliance officers provide critical oversight.
*   **Creative Content Generation:** For marketing copy, blog posts, or design briefs, human editors ensure brand voice, creativity, and emotional resonance.
*   **Complex Decision Support:** In fields like medicine or engineering, agents might provide recommendations, but human experts make the final, critical decisions, often providing feedback on the agent's reasoning.
*   **Data Annotation and Labeling:** Agents can pre-label data, and humans correct errors, creating high-quality datasets for further model training.
*   **Customer Service Escalations:** When an AI chatbot can't resolve an issue, it escalates to a human, who then provides feedback on why the AI failed, improving future interactions.
*   **Continuous Improvement of Agent Prompts/Tools:** By analyzing patterns in human feedback over time, developers can systematically refine agent prompts, adjust tool usage logic, or even fine-tune smaller, specialized models for specific sub-tasks.

This human-in-the-loop approach is not just about correcting errors; it's about building a continuous learning system where your AI agents evolve and improve, becoming more aligned with your specific business needs and delivering higher value over time.


### Resources for Human Feedback and Agent Training

*   **CrewAI Documentation:**
    *   [CrewAI Official Documentation](https://docs.crewai.com/)
    *   [Custom Tools in CrewAI](https://docs.crewai.com/how-to/create-custom-tools/)
    *   [Agent Configuration](https://docs.crewai.com/core-concepts/agents/)

*   **Human-in-the-Loop (HITL) Platforms & Concepts:**
    *   **Argilla:** An open-source platform for building and managing data for LLMs, including human feedback. [Argilla Documentation](https://docs.argilla.io/)
    *   **Label Studio:** Another popular open-source data labeling tool that can be adapted for human feedback. [Label Studio Documentation](https://labelstud.io/guide/)
    *   **Reinforcement Learning from Human Feedback (RLHF):** While complex, understanding the concept is key for advanced training. [Hugging Face Blog on RLHF](https://huggingface.co/blog/rlhf)
    *   **Direct Preference Optimization (DPO):** A simpler, more stable alternative to RLHF for fine-tuning LLMs with human preferences. [Hugging Face Blog on DPO](https://huggingface.co/blog/dpo-llm)

*   **General AI/LLM Resources (2026 Context):**
    *   **Google AI Studio / Gemini API:** For integrating Google's latest models. [Google AI Studio](https://ai.google.dev/)
    *   **OpenAI API:** For integrating OpenAI's latest models. [OpenAI API Documentation](https://platform.openai.com/docs/)
    *   **Hugging Face Transformers & Datasets:** For working with open-source models and datasets, including those for fine-tuning. [Hugging Face](https://huggingface.co/)
    *   **Ollama / vLLM:** For running open-source LLMs locally or on your own infrastructure. [Ollama](https://ollama.com/) [vLLM](https://vllm.ai/)
